In [ ]:
import torch
import functools
import einops
import requests
import pandas as pd
import io
import textwrap
import gc
import json
import numpy as np
from datasets import load_dataset
from sklearn.model_selection import train_test_split
from tqdm import tqdm
from torch import Tensor
from typing import List, Callable
from transformer_lens import HookedTransformer, utils
from transformer_lens.hook_points import HookPoint
from transformers import AutoTokenizer
from jaxtyping import Float, Int
from typing import List, Tuple, Callable
import transformer_lens
import contextlib
import random

In [ ]:
model_name = 'Qwen/Qwen-1_8B-Chat'
device = torch.device('cuda')

model = HookedTransformer.from_pretrained(model_name, device=device)

In [ ]:
QWEN_CHAT_TEMPLATE = """<|im_start|>user
{instruction}<|im_end|>
<|im_start|>assistant
"""

def get_prompt(instruction: str) -> str:
    return QWEN_CHAT_TEMPLATE.format(instruction=instruction)

def get_prompt_tokens(instruction: str) -> torch.Tensor:
    return model.to_tokens(get_prompt(instruction))

tokens_to_consider = 5

In [ ]:
dataset_path = 'artifacts/data/processed/advbench.json'

with open(dataset_path, 'r') as f:
    data = json.load(f)

seed = 42
n = 200
random.seed(seed)
torch.manual_seed(seed)
eval_data = random.sample(data, n)
eval_instructions = [item['instruction'] for item in eval_data]

In [ ]:
# # Load the evaluation results
with open('artifacts/results/residual_paper/k_responses_results/evaluation_passk_advbench.json', 'r') as f:
    evaluation_data = json.load(f)


harm_switch = np.sort([198, 166, 164, 138, 131, 123, 82, 57, 51, 27, 6])

In [ ]:
for eval_item in evaluation_data['responses']:
    if eval_item['instruction_id'] in harm_switch:
        print(eval_item['instruction'])
        print()
        # for key, value in eval_item['responses'].items():
        #     print(value)
        #     print("-"*100)
        # print()
        # print("="*100)

In [ ]:
prompt = eval_instructions[harm_switch[0]]
print(prompt)
prompt_tokens = model.to_tokens(prompt)
logits, cache = model.run_with_cache(prompt_tokens, remove_batch_dim=True)

In [ ]:
probs = logits.softmax(dim=-1)

In [ ]:
QWEN_REFUSAL = ['I', 'Sorry', 'However', 'Unfortunately', 'Instead', 'Alternatively', 'Perhaps']
QWEN_BYPASS = ['Sure', 'Certainly', 'Absolutely', 'Yes', 'Here', 'To', 'First', 'Step', 'Begin', 'Start', 'Create']

QWEN_REFUSAL_TOKENS = [model.to_tokens(word)[0].item() for word in QWEN_REFUSAL]
QWEN_BYPASS_TOKENS = [model.to_tokens(word)[0].item() for word in QWEN_BYPASS]

In [ ]:
QWEN_BYPASS_TOKENS

In [ ]:
print(probs.squeeze(0)[-1, QWEN_BYPASS_TOKENS].sum())
print(probs.squeeze(0)[-1, QWEN_REFUSAL_TOKENS].sum())

In [ ]:
def _generate_with_hooks(
    model: HookedTransformer,
    toks: Int[Tensor, "batch_size seq_len"],
    max_tokens_generated: int = 64,
    fwd_hooks = [],
    temperature: float = 1.0,
    top_p: float = 1.0,
    do_sample: bool = True,
) -> List[str]:
    """
    Generate text with hooks applied, re-evaluating the entire sequence for each new token.
    Supports sampling for diverse responses.
    """
    all_toks = toks.clone()

    with torch.no_grad():
        for _ in range(max_tokens_generated):
            with model.hooks(fwd_hooks=fwd_hooks):
                logits = model(all_toks)
            
            next_token_logits = logits[:, -1, :]
            
            if do_sample and temperature > 0:
                next_token_logits = next_token_logits / temperature
                
                if top_p < 1.0:
                    sorted_logits, sorted_indices = torch.sort(next_token_logits, descending=True)
                    cumulative_probs = torch.cumsum(torch.softmax(sorted_logits, dim=-1), dim=-1)
                    sorted_indices_to_remove = cumulative_probs > top_p
                    sorted_indices_to_remove[..., 1:] = sorted_indices_to_remove[..., :-1].clone()
                    sorted_indices_to_remove[..., 0] = 0
                    
                    for i in range(toks.shape[0]):
                        indices_to_remove = sorted_indices[i][sorted_indices_to_remove[i]]
                        next_token_logits[i, indices_to_remove] = float('-inf')
                
                probs = torch.softmax(next_token_logits, dim=-1)
                next_token = torch.multinomial(probs, num_samples=1)
            else:
                next_token = next_token_logits.argmax(dim=-1).unsqueeze(1)
            
            all_toks = torch.cat([all_toks, next_token], dim=1)

    full_texts = model.to_string(all_toks)
    prompt_texts = model.to_string(toks)

    completions = [
        full[len(prompt):].strip()
        for full, prompt in zip(full_texts, prompt_texts)
    ]
    return completions

In [ ]:
top5_probs, top5_indices = probs.squeeze(0)[-1].topk(5)
top5_tokens = [model.to_string(idx) for idx in top5_indices]
print("Top 5 tokens:")
for i, (prob, token) in enumerate(zip(top5_probs, top5_tokens)):
    print(f"{i+1}. '{token}' (prob: {prob:.4f})")

In [ ]:
prompt = eval_instructions[harm_switch[0]]
prompt

In [ ]:
# Generate completions with empty hooks
i = 3
prompt = eval_instructions[harm_switch[i]]
print(prompt)
prompt = get_prompt(prompt)
toks = model.to_tokens(prompt)

logits, cache = model.run_with_cache(toks, remove_batch_dim=True)
# find top 5 tokens with highest logits
probs = logits.softmax(dim=-1)
top_probs, top_indices = probs.squeeze(0)[-1].topk(5)
top_tokens = [model.to_string(idx) for idx in top_indices]
print(top_tokens, top_probs)
print("-"*100)

for i in range(5):
    generations = _generate_with_hooks(
        model=model,
        toks=toks,
        fwd_hooks=[],
        max_tokens_generated=60,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
    )
    print(generations[0])
    print("-"*50)

In [ ]:
cache.keys()

In [ ]:
# Create tensors to store pre and post residual activations
n_layers = 24  # Based on the cache keys showing blocks.0 through blocks.23
seq_len = cache['blocks.0.hook_resid_pre'].shape[0]
d_model = cache['blocks.0.hook_resid_pre'].shape[1]

# Initialize tensors
cache_pre = torch.zeros(n_layers, seq_len, d_model)
cache_post = torch.zeros(n_layers, seq_len, d_model)

# Fill the tensors with values from cache
for layer in range(n_layers):
    cache_pre[layer] = cache[f'blocks.{layer}.hook_resid_pre']
    cache_post[layer] = cache[f'blocks.{layer}.hook_resid_post']

print(f"cache_pre shape: {cache_pre.shape}")
print(f"cache_post shape: {cache_post.shape}")

In [ ]:
harmful_dir = torch.load('artifacts/residuals/harmful_stack.pt')
harmless_dir = torch.load('artifacts/residuals/harmless_stack.pt')

In [ ]:
# take last 5 tokens of cache_pre 
import matplotlib.pyplot as plt
import torch.nn.functional as F

# Get last 5 tokens of cache_pre for each layer
tokens_to_consider = 5
last_tokens = cache_pre[:, -tokens_to_consider:, :]  # shape: (n_layers, tokens_to_consider, d_model)

# Calculate cosine similarities
cos_sim_harmful = F.cosine_similarity(last_tokens.reshape(-1, last_tokens.size(-1)), 
                                     harmless_dir.reshape(-1, harmless_dir.size(-1)), dim=1)

# Reshape back to (n_layers, tokens_to_consider)
cos_sim_harmful = cos_sim_harmful.reshape(n_layers, tokens_to_consider)

# Create plot
fig, ax = plt.subplots(1, 1, figsize=(8, 6))

# Plot cosine similarity with harmless_dir
im = ax.imshow(cos_sim_harmful.cpu().numpy(), aspect='auto', cmap='viridis', vmin=0, vmax=1)
ax.set_title('Cosine Similarity: Last 5 Tokens vs Harmless Direction')
ax.set_xlabel('Token Position (relative to end)')
ax.set_ylabel('Layer')
ax.set_xticks(range(tokens_to_consider))
ax.set_yticks(range(n_layers))
ax.set_xticklabels([f'Token -{tokens_to_consider-i}' for i in range(tokens_to_consider)])
ax.set_yticklabels([f'{i}' for i in range(n_layers)])
plt.colorbar(im, ax=ax)

# Add text annotations
for i in range(n_layers):
    for j in range(tokens_to_consider):
        text = ax.text(j, i, f'{cos_sim_harmful[i, j]:.3f}', 
                       ha="center", va="center", color="w", fontsize=8)

plt.tight_layout()
plt.show()

print(f"Cosine similarities shape: {cos_sim_harmful.shape}")
print(f"Min similarity: {cos_sim_harmful.min():.4f}")
print(f"Max similarity: {cos_sim_harmful.max():.4f}")
print(f"Mean similarity: {cos_sim_harmful.mean():.4f}")

In [ ]:
# Calculate cosine similarity between harmful and harmless directions
print(f"harmful_dir shape: {harmful_dir.shape}")
print(f"harmless_dir shape: {harmless_dir.shape}")

cos_sim_directions = F.cosine_similarity(harmful_dir.reshape(-1, harmful_dir.size(-1)), 
                                        harmless_dir.reshape(-1, harmless_dir.size(-1)), dim=1)
print(f"cos_sim_directions shape before reshape: {cos_sim_directions.shape}")
print(f"cos_sim_directions size: {cos_sim_directions.numel()}")

# Reshape to (n_layers, tokens_to_consider) since we have 24 layers and 5 tokens
cos_sim_directions = cos_sim_directions.reshape(n_layers, tokens_to_consider)

print(f"Cosine similarity between harmful and harmless directions: {cos_sim_directions.mean():.4f}")

# Plot the cosine similarity between harmful and harmless directions
plt.figure(figsize=(8, 6))
data = cos_sim_directions.cpu().numpy()
im = plt.imshow(data, aspect='auto', cmap='viridis', vmin=-1, vmax=1)
plt.title('Cosine Similarity: Harmful vs Harmless Directions')
plt.xlabel('Token Position (relative to end)')
plt.ylabel('Layer')
plt.xticks(range(tokens_to_consider), [f'Token -{tokens_to_consider-i}' for i in range(tokens_to_consider)])
plt.yticks(range(n_layers), [f'{i}' for i in range(n_layers)])

# Add text annotations with values in each cell
for i in range(n_layers):
    for j in range(tokens_to_consider):
        plt.text(j, i, f'{data[i, j]:.3f}', ha='center', va='center', 
                color='white' if abs(data[i, j]) > 0.5 else 'black', fontsize=8)

plt.colorbar(im, label='Cosine Similarity')
plt.tight_layout()
plt.show()